In [ ]:
# ================================================================
# PHASE 5C — CELL 1
# DRIVE SETUP AND PROJECT DIRECTORIES
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE5_ROOT = (
    PROJECT_ROOT / "phase5_application"
)

PHASE5C_ROOT = (
    PHASE5_ROOT / "phase5C_assistive_application"
)

APP_DIR = PHASE5C_ROOT / "app"
OUTPUT_DIR = PHASE5C_ROOT / "outputs"
SCREENSHOT_DIR = PHASE5C_ROOT / "screenshots"
VERIFICATION_DIR = PHASE5C_ROOT / "verification"

for directory in [
    APP_DIR,
    OUTPUT_DIR,
    SCREENSHOT_DIR,
    VERIFICATION_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 75)
print("PHASE 5C — ASSISTIVE APPLICATION")
print("=" * 75)

print("Project root:", PROJECT_ROOT)
print("Application directory:", APP_DIR)
print("Output directory:", OUTPUT_DIR)

assert PROJECT_ROOT.exists(), (
    "CAMUS_FINAL project directory not found."
)

print("\n✓ Phase 5C directories initialized.")

Mounted at /content/drive
PHASE 5C — ASSISTIVE APPLICATION
Project root: /content/drive/MyDrive/CAMUS_FINAL
Application directory: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/app
Output directory: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/outputs

✓ Phase 5C directories initialized.


In [ ]:
# ================================================================
# PHASE 5C — CELL 2
# VERIFY PHASE 5B COMPLETION
# ================================================================

PHASE5B_COMPLETION_PATH = (
    PHASE5_ROOT
    / "phase5B_integrated_pipeline"
    / "verification"
    / "PHASE5B_COMPLETE.json"
)

assert PHASE5B_COMPLETION_PATH.exists(), (
    "Phase 5B completion record not found. "
    "Please confirm Cell 41 was executed."
)

with open(PHASE5B_COMPLETION_PATH, "r") as f:
    phase5b_record = json.load(f)

assert phase5b_record["status"] == "completed"

assert (
    phase5b_record["matching_frozen_predictions"]
    == 50
)

assert (
    phase5b_record["all_probabilities_reproduced"]
    is True
)

assert (
    phase5b_record["phase2_reproduction_passed"]
    is True
)

print("=" * 75)
print("PHASE 5B — FROZEN INTEGRATION VERIFIED")
print("=" * 75)

print(
    "Matching classifications:",
    phase5b_record["matching_frozen_predictions"],
    "/50"
)

print(
    "Maximum probability difference:",
    phase5b_record["maximum_probability_difference"]
)

print(
    "\n✓ Phase 5C may safely reuse "
    "the verified frozen pipeline."
)

PHASE 5B — FROZEN INTEGRATION VERIFIED
Matching classifications: 50 /50
Maximum probability difference: 4.440892098500626e-16

✓ Phase 5C may safely reuse the verified frozen pipeline.


In [ ]:
# ================================================================
# PHASE 5C — CELL 3
# APPLICATION DEPENDENCIES
# ================================================================

%pip install -q gradio nibabel opencv-python-headless

import gradio as gr
import nibabel as nib
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import joblib

from scipy.ndimage import binary_fill_holes
from sklearn.pipeline import Pipeline

print("=" * 75)
print("PHASE 5C — DEPENDENCIES")
print("=" * 75)

print("Gradio:", gr.__version__)
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("Nibabel:", nib.__version__)

print("\n✓ Application dependencies available.")

PHASE 5C — DEPENDENCIES
Gradio: 6.29.0
TensorFlow: 2.21.0
OpenCV: 4.14.0
Nibabel: 5.4.2

✓ Application dependencies available.


In [ ]:
# ================================================================
# PHASE 5C — CELL 4
# EXTRACT VERIFIED PHASE 5B INFERENCE FUNCTIONS
# ================================================================

import json
from pathlib import Path

notebook_matches = list(
    Path("/content/drive/MyDrive").rglob(
        "Phase5B_Integrated_Frozen_Inference_Pipeline.ipynb"
    )
)

assert len(notebook_matches) == 1, (
    "Expected exactly one Phase 5B notebook. "
    f"Found: {notebook_matches}"
)

SOURCE_NOTEBOOK = notebook_matches[0]

with open(SOURCE_NOTEBOOK, encoding="utf-8") as f:
    notebook = json.load(f)

# These are the verified Phase 5B implementation cells.
required_cells = {
    "CELL 30": "def load_nifti_2d(",
    "CELL 31": "def dice_coefficient(",
    "CELL 33": "def analyze_lv_frame(",
    "CELL 38": "def analyze_a4c_pair("
}

selected_sources = []

for cell_label, required_function in required_cells.items():

    matches = [
        "".join(cell.get("source", []))
        for cell in notebook["cells"]
        if cell.get("cell_type") == "code"
        and required_function in "".join(cell.get("source", []))
    ]

    assert len(matches) == 1, (
        f"Could not uniquely locate {cell_label}: "
        f"{required_function}"
    )

    selected_sources.append(matches[0])

MODULE_PATH = APP_DIR / "frozen_inference.py"

module_header = """
# Frozen inference implementation extracted from Phase 5B.
# Do not modify preprocessing, segmentation or geometry methods.

import numpy as np
import pandas as pd
import cv2
import nibabel as nib
import tensorflow as tf
import joblib

from pathlib import Path
from scipy.ndimage import binary_fill_holes
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path("/content/drive/MyDrive/CAMUS_FINAL")

FROZEN_SEGMENTATION_MODEL_PATH = (
    PROJECT_ROOT
    / "phase1_segmentation"
    / "phase1F_final_model"
    / "models"
    / "final_lv_segmentation_model.keras"
)

PHASE3_CLASSIFIER_ARTIFACT_PATH = (
    PROJECT_ROOT
    / "phase3_classification"
    / "phase3C_classifier_development"
    / "models"
    / "phase3C_selected_logistic_regression.joblib"
)

"""

# Add the classifier-loading statement after the model-loading
# implementation and before the patient-level inference function.
classifier_loader = """
frozen_phase3_classifier = joblib.load(
    PHASE3_CLASSIFIER_ARTIFACT_PATH
)

assert isinstance(frozen_phase3_classifier, Pipeline)

assert list(frozen_phase3_classifier.classes_) == [
    "Higher_EF",
    "Intermediate_EF",
    "Lower_EF"
]
"""

module_text = (
    module_header
    + "\n\n"
    + selected_sources[0]
    + "\n\n"
    + selected_sources[1]
    + "\n\n"
    + selected_sources[2]
    + "\n\n"
    + classifier_loader
    + "\n\n"
    + selected_sources[3]
)

# Check syntax before saving.
compile(module_text, str(MODULE_PATH), "exec")

MODULE_PATH.write_text(module_text, encoding="utf-8")

print("=" * 75)
print("FROZEN PHASE 5B MODULE CREATED")
print("=" * 75)
print("Source:", SOURCE_NOTEBOOK)
print("Module:", MODULE_PATH)
print("Functions extracted:", len(selected_sources))
print("Syntax verification: PASSED")

FROZEN PHASE 5B MODULE CREATED
Source: /content/drive/MyDrive/Colab Notebooks/Phase5B_Integrated_Frozen_Inference_Pipeline.ipynb
Module: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/app/frozen_inference.py
Functions extracted: 4
Syntax verification: PASSED


In [ ]:
# ================================================================
# PHASE 5C — CELL 5
# LOAD REUSABLE INFERENCE MODULE
# ================================================================

import sys
import importlib

sys.path.insert(0, str(APP_DIR))

if "frozen_inference" in sys.modules:
    del sys.modules["frozen_inference"]

import frozen_inference as inference

assert inference.frozen_segmentation_model.count_params() == 7759521

assert list(inference.frozen_phase3_classifier.classes_) == [
    "Higher_EF",
    "Intermediate_EF",
    "Lower_EF"
]

print("=" * 75)
print("PHASE 5C — FROZEN PIPELINE LOADED")
print("=" * 75)

print(
    "Segmentation parameters:",
    inference.frozen_segmentation_model.count_params()
)

print(
    "Classifier classes:",
    inference.frozen_phase3_classifier.classes_
)

print("\n✓ Ready for application integration.")

FROZEN PHASE 2 MEASUREMENT FUNCTIONS

✓ 256×256 stretch preprocessing
✓ Post-resize per-image min-max normalization
✓ Anisotropic effective spacing
✓ Threshold >= 0.50
✓ 8-connected largest component
✓ Binary hole filling
✓ Physical LV area
✓ Exact PCA major-axis length
✓ Single-plane area-length volume
FROZEN SEGMENTATION MODEL LOADED

Parameters: 7,759,521
Input shape : (None, 256, 256, 1)
Output shape: (None, 256, 256, 1)

✓ Frozen Phase 1 model ready for inference.
✓ Single-frame frozen inference function ready.
COMPLETE FROZEN A4C PIPELINE

ED/ES input
   ↓
Frozen LV segmentation
   ↓
Physical LV geometry
   ↓
EDV / ESV / SV / EF
   ↓
Frozen EF-band classifier
   ↓
EF band + uncalibrated confidence

✓ analyze_a4c_pair() ready.
PHASE 5C — FROZEN PIPELINE LOADED
Segmentation parameters: 7759521
Classifier classes: ['Higher_EF' 'Intermediate_EF' 'Lower_EF']

✓ Ready for application integration.


In [ ]:
# ================================================================
# PHASE 5C — CELL 6A
# RESTORE CAMUS DATASET FROM GOOGLE DRIVE
# ================================================================

from pathlib import Path
import zipfile
import shutil
import time

OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR / "download"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)

DATASET_PATH = (
    INNER_EXTRACT_DIR / "database_nifti"
)

print("=" * 75)
print("PHASE 5C — CAMUS DATASET RESTORATION")
print("=" * 75)

start_time = time.time()

# Check whether the dataset already exists.
if DATASET_PATH.exists():

    print("Dataset already extracted.")

else:

    assert OUTER_ZIP_PATH.exists(), (
        f"Dataset archive not found: {OUTER_ZIP_PATH}"
    )

    OUTER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\nExtracting outer archive...")

    with zipfile.ZipFile(OUTER_ZIP_PATH, "r") as archive:
        archive.extractall(OUTER_EXTRACT_DIR)

    assert INNER_ARCHIVE_PATH.exists(), (
        "Inner CAMUS archive not found after extraction."
    )

    INNER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    print("Extracting inner CAMUS archive...")

    if zipfile.is_zipfile(INNER_ARCHIVE_PATH):

        with zipfile.ZipFile(
            INNER_ARCHIVE_PATH, "r"
        ) as archive:

            archive.extractall(
                INNER_EXTRACT_DIR
            )

    else:

        shutil.unpack_archive(
            str(INNER_ARCHIVE_PATH),
            str(INNER_EXTRACT_DIR)
        )

assert DATASET_PATH.exists(), (
    "CAMUS database_nifti directory was not found."
)

patient_dirs = sorted(
    p for p in DATASET_PATH.iterdir()
    if p.is_dir() and p.name.startswith("patient")
)

print("\nDataset path:", DATASET_PATH)
print("Patient directories:", len(patient_dirs))
print(
    "Elapsed time:",
    round(time.time() - start_time, 2),
    "seconds"
)

assert len(patient_dirs) == 500, (
    f"Expected 500 patients, found {len(patient_dirs)}."
)

print("\n✓ CAMUS dataset restored successfully.")

PHASE 5C — CAMUS DATASET RESTORATION

Extracting outer archive...
Extracting inner CAMUS archive...

Dataset path: /content/camus_data_final/database_nifti
Patient directories: 500
Elapsed time: 137.04 seconds

✓ CAMUS dataset restored successfully.


In [ ]:
# ================================================================
# PHASE 5C — CELL 6
# INDEPENDENT MODULE SMOKE TEST
# ================================================================

# Dataset extraction is not needed for this step.
# We use the frozen patient0173 quantitative result as a reference
# only after locating the source ED/ES images.

DATASET_PATH = Path(
    "/content/camus_data_final/database_nifti"
)

if not DATASET_PATH.exists():
    print(
        "CAMUS dataset not currently extracted. "
        "Restore the dataset using the original "
        "Phase 5B extraction cell before this test."
    )
else:
    patient_id = "patient0173"
    patient_dir = DATASET_PATH / patient_id

    ed_image, ed_spacing = inference.load_nifti_2d(
        patient_dir / f"{patient_id}_4CH_ED.nii.gz"
    )

    es_image, es_spacing = inference.load_nifti_2d(
        patient_dir / f"{patient_id}_4CH_ES.nii.gz"
    )

    result = inference.analyze_a4c_pair(
        ed_image,
        es_image,
        ed_spacing,
        es_spacing
    )

    measurements = result["quantitative_measurements"]
    classification = result["classification"]

    assert np.isclose(
        measurements["EF_pct"],
        60.649136,
        atol=1e-6
    )

    assert (
        classification["estimated_EF_band"]
        == "Higher_EF"
    )

    assert np.isclose(
        classification["model_confidence_score"],
        0.695430,
        atol=1e-6
    )

    print("=" * 75)
    print("PHASE 5C — MODULE SMOKE TEST")
    print("=" * 75)

    print("Patient:", patient_id)
    print("EDV:", measurements["EDV_ml"])
    print("ESV:", measurements["ESV_ml"])
    print("EF:", measurements["EF_pct"])
    print("EF band:", classification["estimated_EF_band"])
    print(
        "Model confidence:",
        classification["model_confidence_score"]
    )

    print("\n✓ Frozen module smoke test PASSED.")

PHASE 5C — MODULE SMOKE TEST
Patient: patient0173
EDV: 80.02545550461342
ESV: 31.490708497854754
EF: 60.64913557906667
EF band: Higher_EF
Model confidence: 0.6954298611661843

✓ Frozen module smoke test PASSED.


In [ ]:
# ================================================================
# PHASE 5C — CELL 7
# SEGMENTATION VISUALIZATION
# ================================================================

import numpy as np
import cv2
import matplotlib.pyplot as plt

def create_lv_overlay(processed_image, predicted_mask):
    """
    Display the original processed grayscale frame
    with a green LV cavity boundary.
    """

    image = np.asarray(processed_image, dtype=np.float32)
    mask = np.asarray(predicted_mask, dtype=np.uint8)

    assert image.ndim == 2
    assert image.shape == mask.shape

    image_uint8 = np.clip(image * 255, 0, 255).astype(np.uint8)

    rgb = cv2.cvtColor(
        image_uint8,
        cv2.COLOR_GRAY2RGB
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    cv2.drawContours(
        rgb,
        contours,
        -1,
        (0, 255, 100),
        2
    )

    return rgb


print("✓ Segmentation overlay function ready.")

✓ Segmentation overlay function ready.


In [ ]:
# ================================================================
# PHASE 5C — CELL 8
# APPLICATION INFERENCE HANDLER
# ================================================================

import pandas as pd

def run_assistive_analysis(ed_file, es_file):

    if ed_file is None or es_file is None:
        raise gr.Error(
            "Please upload both ED and ES NIfTI frames."
        )

    try:
        # --------------------------------------------------------
        # Load native ED/ES frames and physical spacing
        # --------------------------------------------------------

        ed_image, ed_spacing = inference.load_nifti_2d(
            ed_file
        )

        es_image, es_spacing = inference.load_nifti_2d(
            es_file
        )

        # --------------------------------------------------------
        # Run complete frozen pipeline
        # --------------------------------------------------------

        result = inference.analyze_a4c_pair(
            ed_image,
            es_image,
            ed_spacing,
            es_spacing
        )

        ed_seg = result["segmentation"]["ED"]
        es_seg = result["segmentation"]["ES"]

        metrics = result["quantitative_measurements"]
        classification = result["classification"]

        # --------------------------------------------------------
        # Create visualizations
        # --------------------------------------------------------

        ed_overlay = create_lv_overlay(
            ed_seg["processed_image"],
            ed_seg["predicted_mask"]
        )

        es_overlay = create_lv_overlay(
            es_seg["processed_image"],
            es_seg["predicted_mask"]
        )

        # --------------------------------------------------------
        # Measurements
        # --------------------------------------------------------

        measurement_table = pd.DataFrame([
            ["ED area (mm²)", metrics["ED_area_mm2"]],
            ["ES area (mm²)", metrics["ES_area_mm2"]],
            ["ED length (mm)", metrics["ED_length_mm"]],
            ["ES length (mm)", metrics["ES_length_mm"]],
            ["EDV (mL)", metrics["EDV_ml"]],
            ["ESV (mL)", metrics["ESV_ml"]],
            ["SV (mL)", metrics["SV_ml"]],
            ["EF (%)", metrics["EF_pct"]],
        ], columns=["Measurement", "Value"])

        measurement_table["Value"] = (
            measurement_table["Value"].round(3)
        )

        # --------------------------------------------------------
        # EF classification and confidence
        # --------------------------------------------------------

        ef_band = classification["estimated_EF_band"]

        scores = classification["class_scores"]

        score_table = pd.DataFrame([
            [class_name, round(score, 6)]
            for class_name, score in scores.items()
        ], columns=["EF band", "Uncalibrated score"])

        summary = (
            f"### Estimated EF Band: {ef_band}\n\n"
            f"**Estimated EF:** {metrics['EF_pct']:.2f}%  \n"
            f"**Maximum classifier score:** "
            f"{classification['model_confidence_score']:.4f}\n\n"
            "**Important:** This score is uncalibrated and "
            "does not establish patient-level reliability. "
            "This is a research-only, non-diagnostic "
            "assistive application."
        )

        return (
            ed_overlay,
            es_overlay,
            measurement_table,
            summary,
            score_table
        )

    except gr.Error:
        raise

    except Exception as exc:
        raise gr.Error(
            f"Unable to analyze the supplied ED/ES files: {exc}"
        )


print("✓ Application inference handler ready.")

✓ Application inference handler ready.


In [ ]:
# ================================================================
# PHASE 5C — CELL 9
# GRADIO ASSISTIVE APPLICATION
# ================================================================

with gr.Blocks(
    title="A4C LV Analysis — ASCEND Research Prototype"
) as demo:

    gr.Markdown(
        """
        # A4C Left Ventricular Analysis
        **ASCEND Research Lab | Computer Vision Assistive System**

        Upload two identified A4C echocardiography frames
        corresponding to **End Diastole (ED)** and
        **End Systole (ES)**.

        The files must be `.nii` or `.nii.gz` images with
        valid physical pixel-spacing metadata.

        **Research prototype — not intended for clinical
        diagnosis or treatment decisions.**
        """
    )

    with gr.Row():

        ed_input = gr.File(
            label="Upload ED Frame",
            file_types=[".nii", ".gz"],
            type="filepath"
        )

        es_input = gr.File(
            label="Upload ES Frame",
            file_types=[".nii", ".gz"],
            type="filepath"
        )

    analyze_button = gr.Button(
        "Analyze ED/ES Frames",
        variant="primary"
    )

    gr.Markdown("## Predicted LV Segmentation")

    with gr.Row():

        ed_output = gr.Image(
            label="ED — LV Boundary",
            type="numpy",
            interactive=False
        )

        es_output = gr.Image(
            label="ES — LV Boundary",
            type="numpy",
            interactive=False
        )

    gr.Markdown("## Quantitative LV Measurements")

    measurement_output = gr.Dataframe(
        headers=["Measurement", "Value"],
        interactive=False
    )

    gr.Markdown("## Functional Classification")

    classification_output = gr.Markdown()

    confidence_output = gr.Dataframe(
        headers=["EF band", "Uncalibrated score"],
        interactive=False
    )

    analyze_button.click(
        fn=run_assistive_analysis,
        inputs=[ed_input, es_input],
        outputs=[
            ed_output,
            es_output,
            measurement_output,
            classification_output,
            confidence_output
        ]
    )

    gr.Markdown(
        """
        ---
        **Limitations:** Requires correctly identified A4C
        ED/ES frames and valid physical spacing. Does not
        automatically identify cardiac views or ED/ES phases.
        EF bands are research-defined functional categories,
        not clinical diagnoses. Confidence scores are not
        calibrated probabilities.
        """
    )

print("✓ Gradio interface constructed.")

✓ Gradio interface constructed.


In [ ]:
# ================================================================
# PHASE 5C — CELL 12
# PREPARE DOWNLOADABLE CAMUS DEMONSTRATION FILES
# ================================================================

from pathlib import Path
import shutil
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

patient_id = "patient0173"

patient_dir = DATASET_PATH / patient_id

demo_folder = (
    PHASE5C_ROOT
    / "demo_patient0173"
)

demo_folder.mkdir(
    parents=True,
    exist_ok=True
)

for phase in ["ED", "ES"]:

    source_path = (
        patient_dir
        / f"{patient_id}_4CH_{phase}.nii.gz"
    )

    assert source_path.exists()

    # Copy the original NIfTI image.
    destination_path = (
        demo_folder
        / source_path.name
    )

    shutil.copy2(
        source_path,
        destination_path
    )

    # Create a preview image.
    image, spacing = inference.load_nifti_2d(
        source_path
    )

    preview_path = (
        demo_folder
        / f"{patient_id}_4CH_{phase}_preview.png"
    )

    plt.imsave(
        preview_path,
        image,
        cmap="gray"
    )

    print(f"{phase} image prepared.")
    print("Physical spacing:", spacing)

# Create a single ZIP containing all files.
zip_path = (
    PHASE5C_ROOT
    / "patient0173_demo_files.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:

    for file_path in demo_folder.iterdir():
        archive.write(
            file_path,
            arcname=file_path.name
        )

print("\nDemonstration package created:")
print(zip_path)

files.download(str(zip_path))

ED image prepared.
Physical spacing: (0.30799999833106995, 0.30799999833106995)
ES image prepared.
Physical spacing: (0.30799999833106995, 0.30799999833106995)

Demonstration package created:
/content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/patient0173_demo_files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ================================================================
# PHASE 5C — CELL 10
# LAUNCH GRADIO APPLICATION IN GOOGLE COLAB
# ================================================================

# Close any previous Gradio server
try:
    demo.close()
except Exception:
    pass

# Launch with a temporary public link
demo.launch(
    share=True,
    debug=True,
    show_error=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://caf3b6ed516fa91b68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://caf3b6ed516fa91b68.gradio.live


In [ ]:
# ================================================================
# PHASE 5C — CELL 11
# DEMONSTRATION PATIENT FILES
# ================================================================

patient_id = "patient0173"

patient_dir = DATASET_PATH / patient_id

demo_ed_path = (
    patient_dir / f"{patient_id}_4CH_ED.nii.gz"
)

demo_es_path = (
    patient_dir / f"{patient_id}_4CH_ES.nii.gz"
)

assert demo_ed_path.exists()
assert demo_es_path.exists()

print("ED file:", demo_ed_path)
print("ES file:", demo_es_path)

print("\nExpected demonstration results:")
print("EDV: approximately 80.025 mL")
print("ESV: approximately 31.491 mL")
print("EF: approximately 60.649%")
print("Estimated EF band: Higher_EF")

ED file: /content/camus_data_final/database_nifti/patient0173/patient0173_4CH_ED.nii.gz
ES file: /content/camus_data_final/database_nifti/patient0173/patient0173_4CH_ES.nii.gz

Expected demonstration results:
EDV: approximately 80.025 mL
ESV: approximately 31.491 mL
EF: approximately 60.649%
Estimated EF band: Higher_EF


In [ ]:
# ================================================================
# PHASE 5C — CELL 13
# SAVE APPLICATION DEMONSTRATION EVIDENCE
# ================================================================

import json
from datetime import datetime, timezone
from PIL import Image

# Run the same function used by the Gradio interface.
demo_outputs = run_assistive_analysis(
    str(demo_ed_path),
    str(demo_es_path)
)

ed_overlay, es_overlay, measurements, summary, scores = demo_outputs

# Save segmentation visualizations.
ed_overlay_path = OUTPUT_DIR / "patient0173_ED_overlay.png"
es_overlay_path = OUTPUT_DIR / "patient0173_ES_overlay.png"

Image.fromarray(ed_overlay).save(ed_overlay_path)
Image.fromarray(es_overlay).save(es_overlay_path)

# Save numerical outputs.
measurements_path = OUTPUT_DIR / "patient0173_measurements.csv"
scores_path = OUTPUT_DIR / "patient0173_class_scores.csv"

measurements.to_csv(measurements_path, index=False)
scores.to_csv(scores_path, index=False)

# Confirm frozen reference results.
measurement_lookup = dict(
    zip(measurements["Measurement"], measurements["Value"])
)

assert np.isclose(
    measurement_lookup["EF (%)"],
    60.649,
    atol=0.001
)

assert "Higher_EF" in summary

# Save completion record.
completion_record = {
    "phase": "5C",
    "status": "completed",
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "application_framework": "Gradio",
    "demo_patient": "patient0173",
    "ed_overlay_generated": True,
    "es_overlay_generated": True,
    "quantitative_measurements_generated": True,
    "ef_band_classification_generated": True,
    "uncalibrated_scores_displayed": True,
    "non_diagnostic_disclaimer_included": True,
    "frozen_models_modified": False,
    "scope": (
        "Research demonstration with paired identified "
        "A4C ED/ES NIfTI images."
    )
}

completion_path = VERIFICATION_DIR / "PHASE5C_COMPLETE.json"

with open(completion_path, "w") as f:
    json.dump(completion_record, f, indent=2)

print("=" * 75)
print("PHASE 5C — DEMONSTRATION SAVED")
print("=" * 75)

print("ED overlay:", ed_overlay_path)
print("ES overlay:", es_overlay_path)
print("Measurements:", measurements_path)
print("Class scores:", scores_path)
print("Completion record:", completion_path)

print("\n✓ Phase 5C demonstration evidence preserved.")

PHASE 5C — DEMONSTRATION SAVED
ED overlay: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/outputs/patient0173_ED_overlay.png
ES overlay: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/outputs/patient0173_ES_overlay.png
Measurements: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/outputs/patient0173_measurements.csv
Class scores: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/outputs/patient0173_class_scores.csv
Completion record: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5C_assistive_application/verification/PHASE5C_COMPLETE.json

✓ Phase 5C demonstration evidence preserved.


In [ ]:
# ================================================================
# PHASE 5D — CELL 14
# FINAL SYSTEM VERIFICATION SETUP
# ================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone

PHASE5D_ROOT = (
    PROJECT_ROOT
    / "phase5_application"
    / "phase5D_final_verification"
)

PHASE5D_TABLES = PHASE5D_ROOT / "tables"
PHASE5D_RECORDS = PHASE5D_ROOT / "verification"
PHASE5D_REPORTS = PHASE5D_ROOT / "reports"

for folder in [
    PHASE5D_TABLES,
    PHASE5D_RECORDS,
    PHASE5D_REPORTS
]:
    folder.mkdir(parents=True, exist_ok=True)

assert (
    VERIFICATION_DIR / "PHASE5C_COMPLETE.json"
).exists()

assert (
    PHASE5_ROOT
    / "phase5B_integrated_pipeline"
    / "verification"
    / "PHASE5B_COMPLETE.json"
).exists()

print("=" * 80)
print("PHASE 5D — FINAL SYSTEM VERIFICATION")
print("=" * 80)

print("Phase 5B completion record: FOUND")
print("Phase 5C completion record: FOUND")
print("Verification directory:", PHASE5D_ROOT)

print("\n✓ Ready to begin final system tests.")

PHASE 5D — FINAL SYSTEM VERIFICATION
Phase 5B completion record: FOUND
Phase 5C completion record: FOUND
Verification directory: /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5D_final_verification

✓ Ready to begin final system tests.


In [ ]:
# ================================================================
# PHASE 5D — CELL 15
# VALID INPUT AND FROZEN REFERENCE CONSISTENCY
# ================================================================

phase2_reference_path = (
    PROJECT_ROOT
    / "phase2_quantitative"
    / "phase2B_final_evaluation"
    / "tables"
    / "final_test_quantitative_measurements.csv"
)

reference_df = pd.read_csv(phase2_reference_path)

patient_id = "patient0173"

reference_row = reference_df.loc[
    reference_df["patient_id"] == patient_id
].iloc[0]

app_outputs = run_assistive_analysis(
    str(demo_ed_path),
    str(demo_es_path)
)

ed_overlay, es_overlay, measurements, summary, scores = app_outputs

measurement_lookup = dict(
    zip(
        measurements["Measurement"],
        measurements["Value"]
    )
)

comparison_map = {
    "ED area (mm²)": "ED_PRED_area_mm2",
    "ES area (mm²)": "ES_PRED_area_mm2",
    "ED length (mm)": "ED_PRED_length_mm",
    "ES length (mm)": "ES_PRED_length_mm",
    "EDV (mL)": "ED_PRED_volume_ml",
    "ESV (mL)": "ES_PRED_volume_ml",
    "SV (mL)": "PRED_SV_ml",
    "EF (%)": "PRED_EF_pct"
}

verification_rows = []

for app_name, frozen_column in comparison_map.items():

    app_value = float(measurement_lookup[app_name])
    frozen_value = float(reference_row[frozen_column])

    # The Gradio table intentionally rounds to 3 decimals.
    difference = abs(app_value - frozen_value)

    passed = bool(
        np.isclose(
            app_value,
            frozen_value,
            atol=0.00051,
            rtol=0
        )
    )

    verification_rows.append({
        "measurement": app_name,
        "frozen_value": frozen_value,
        "application_value": app_value,
        "absolute_difference": difference,
        "passed": passed
    })

phase5d_numerical_df = pd.DataFrame(verification_rows)

display(phase5d_numerical_df)

numerical_passed = bool(
    phase5d_numerical_df["passed"].all()
)

assert ed_overlay.shape == (256, 256, 3)
assert es_overlay.shape == (256, 256, 3)

assert "Higher_EF" in summary
assert "non-diagnostic" in summary.lower()
assert "uncalibrated" in summary.lower()

assert len(scores) == 3

print("\nNumerical verification:", numerical_passed)
print("Segmentation overlays: PASSED")
print("EF-band output: PASSED")
print("Disclaimer and confidence wording: PASSED")

assert numerical_passed

print("\n✓ VALID INPUT SYSTEM TEST PASSED.")

,measurement,frozen_value,application_value,absolute_difference,passed
0,ED area (mm²),2714.265708,2714.266,0.000292,True
1,ES area (mm²),1520.238767,1520.239,0.000233,True
2,ED length (mm),78.143962,78.144,0.000038,True
3,ES length (mm),62.295982,62.296,0.000018,True
4,EDV (mL),80.025456,80.025,0.000456,True
5,ESV (mL),31.490708,31.491,0.000292,True
6,SV (mL),48.534747,48.535,0.000253,True
7,EF (%),60.649136,60.649,0.000136,True



Numerical verification: True
Segmentation overlays: PASSED
EF-band output: PASSED
Disclaimer and confidence wording: PASSED

✓ VALID INPUT SYSTEM TEST PASSED.


In [ ]:
# ================================================================
# PHASE 5D — CELL 16
# INPUT ERROR-HANDLING TESTS
# ================================================================

import tempfile
from pathlib import Path

test_cases = [
    {
        "name": "Missing ED input",
        "ed": None,
        "es": str(demo_es_path)
    },
    {
        "name": "Missing ES input",
        "ed": str(demo_ed_path),
        "es": None
    },
    {
        "name": "Both inputs missing",
        "ed": None,
        "es": None
    }
]

# Create a corrupt file to test NIfTI loading failure.
with tempfile.TemporaryDirectory() as temp_dir:

    invalid_path = Path(temp_dir) / "invalid_frame.nii.gz"

    invalid_path.write_bytes(
        b"This is not a valid NIfTI image."
    )

    test_cases.append({
        "name": "Corrupt NIfTI input",
        "ed": str(invalid_path),
        "es": str(demo_es_path)
    })

    error_test_results = []

    for case in test_cases:

        try:
            run_assistive_analysis(
                case["ed"],
                case["es"]
            )

            rejected = False
            error_message = "Input was unexpectedly accepted"

        except Exception as exc:

            rejected = True
            error_message = str(exc)

        error_test_results.append({
            "test": case["name"],
            "rejected": rejected,
            "message": error_message[:250]
        })

error_test_df = pd.DataFrame(error_test_results)

display(error_test_df)

assert error_test_df["rejected"].all(), (
    "At least one invalid-input test failed."
)

print("\n✓ MISSING AND CORRUPT INPUT TESTS PASSED.")

,test,rejected,message
0,Missing ED input,True,'Please upload both ED and ES NIfTI frames.'
1,Missing ES input,True,'Please upload both ED and ES NIfTI frames.'
2,Both inputs missing,True,'Please upload both ED and ES NIfTI frames.'
3,Corrupt NIfTI input,True,'Unable to analyze the supplied ED/ES files: F...



✓ MISSING AND CORRUPT INPUT TESTS PASSED.


In [ ]:
# ================================================================
# PHASE 5D — CELL 17
# VALIDATE INPUT IMAGE AND PHYSICAL SPACING
# ================================================================

import numpy as np
import nibabel as nib

def validate_a4c_input(image, spacing, label):
    """
    Validate a 2D A4C frame and physical pixel spacing.

    Does not validate whether the image is truly A4C
    or whether ED/ES phase labels are correct.
    """

    image = np.asarray(image)

    if image.ndim != 2:
        raise ValueError(
            f"{label}: Expected a 2D image."
        )

    if min(image.shape) < 2:
        raise ValueError(
            f"{label}: Image dimensions are invalid."
        )

    if not np.all(np.isfinite(image)):
        raise ValueError(
            f"{label}: Image contains NaN or infinity."
        )

    if not np.issubdtype(image.dtype, np.number):
        raise ValueError(
            f"{label}: Image must contain numeric pixels."
        )

    if not np.isfinite(image.min()) or image.max() <= image.min():
        raise ValueError(
            f"{label}: Image has no usable intensity variation."
        )

    try:
        spacing = np.asarray(spacing, dtype=float)
    except (TypeError, ValueError):
        raise ValueError(
            f"{label}: Invalid physical pixel spacing."
        )

    if spacing.shape != (2,):
        raise ValueError(
            f"{label}: Expected exactly two spacing values."
        )

    if not np.all(np.isfinite(spacing)):
        raise ValueError(
            f"{label}: Physical spacing must be finite."
        )

    if np.any(spacing <= 0):
        raise ValueError(
            f"{label}: Physical spacing must be positive."
        )

    return True


print("✓ A4C input validation function ready.")

✓ A4C input validation function ready.


In [ ]:
# ================================================================
# PHASE 5D — CELL 18
# VALIDATED APPLICATION HANDLER
# ================================================================

original_run_assistive_analysis = run_assistive_analysis

def validated_run_assistive_analysis(ed_file, es_file):

    if ed_file is None or es_file is None:
        raise gr.Error(
            "Please upload both ED and ES NIfTI frames."
        )

    try:
        ed_image, ed_spacing = inference.load_nifti_2d(
            ed_file
        )

        es_image, es_spacing = inference.load_nifti_2d(
            es_file
        )

        validate_a4c_input(
            ed_image,
            ed_spacing,
            "ED"
        )

        validate_a4c_input(
            es_image,
            es_spacing,
            "ES"
        )

    except Exception as exc:
        raise gr.Error(
            f"Input validation failed: {exc}"
        )

    return original_run_assistive_analysis(
        ed_file,
        es_file
    )


# Rebind the existing button to the validated function.
# Gradio registers event handlers when .click() is called,
# so rebuilding the Blocks interface is safer than adding
# another click event to the same button.

run_assistive_analysis = validated_run_assistive_analysis

print("✓ Validated application handler ready.")
print(
    "IMPORTANT: Rerun Cell 9 to rebuild the Gradio interface, "
    "then relaunch Cell 10."
)

✓ Validated application handler ready.
IMPORTANT: Rerun Cell 9 to rebuild the Gradio interface, then relaunch Cell 10.


In [ ]:
# ================================================================
# PHASE 5D — CELL 19
# VALIDATION TESTS
# ================================================================

valid_image = np.array(
    [[0.0, 1.0],
     [0.5, 0.8]],
    dtype=np.float32
)

validation_cases = [
    ("Valid input", valid_image, (0.3, 0.3), True),
    ("Zero spacing", valid_image, (0.0, 0.3), False),
    ("Negative spacing", valid_image, (-0.3, 0.3), False),
    ("NaN spacing", valid_image, (np.nan, 0.3), False),
    ("Infinite spacing", valid_image, (np.inf, 0.3), False),
    ("Missing spacing", valid_image, None, False),
    ("Constant image", np.ones((2, 2)), (0.3, 0.3), False),
    ("NaN image", np.array([[0, 1], [np.nan, 2]]), (0.3, 0.3), False),
]

validation_results = []

for name, image, spacing, expected_valid in validation_cases:

    try:
        validate_a4c_input(image, spacing, name)
        actual_valid = True
    except (ValueError, TypeError):
        actual_valid = False

    validation_results.append({
        "test": name,
        "expected_valid": expected_valid,
        "actual_valid": actual_valid,
        "passed": expected_valid == actual_valid
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

assert validation_df["passed"].all()

print("\n✓ ALL INPUT VALIDATION TESTS PASSED.")

,test,expected_valid,actual_valid,passed
0,Valid input,True,True,True
1,Zero spacing,False,False,True
2,Negative spacing,False,False,True
3,NaN spacing,False,False,True
4,Infinite spacing,False,False,True
5,Missing spacing,False,False,True
6,Constant image,False,False,True
7,NaN image,False,False,True



✓ ALL INPUT VALIDATION TESTS PASSED.


In [ ]:
# ================================================================
# PHASE 5D — CELL 20
# FINAL VALIDATED APPLICATION REGRESSION TEST
# ================================================================

# Run the updated application handler.
final_outputs = run_assistive_analysis(
    str(demo_ed_path),
    str(demo_es_path)
)

final_ed_overlay, final_es_overlay, final_measurements, \
final_summary, final_scores = final_outputs

# Compare against the frozen reference values.
reference_row = reference_df.loc[
    reference_df["patient_id"] == "patient0173"
].iloc[0]

measurement_lookup = dict(
    zip(
        final_measurements["Measurement"],
        final_measurements["Value"]
    )
)

comparison_map = {
    "ED area (mm²)": "ED_PRED_area_mm2",
    "ES area (mm²)": "ES_PRED_area_mm2",
    "ED length (mm)": "ED_PRED_length_mm",
    "ES length (mm)": "ES_PRED_length_mm",
    "EDV (mL)": "ED_PRED_volume_ml",
    "ESV (mL)": "ES_PRED_volume_ml",
    "SV (mL)": "PRED_SV_ml",
    "EF (%)": "PRED_EF_pct"
}

regression_results = []

for display_name, frozen_column in comparison_map.items():

    actual = float(measurement_lookup[display_name])
    expected = float(reference_row[frozen_column])

    difference = abs(actual - expected)

    passed = bool(
        np.isclose(
            actual,
            expected,
            atol=0.00051,
            rtol=0
        )
    )

    regression_results.append({
        "measurement": display_name,
        "frozen_value": expected,
        "validated_app_value": actual,
        "absolute_difference": difference,
        "passed": passed
    })

final_regression_df = pd.DataFrame(regression_results)

display(final_regression_df)

assert final_regression_df["passed"].all()
assert final_ed_overlay.shape == (256, 256, 3)
assert final_es_overlay.shape == (256, 256, 3)

assert "Higher_EF" in final_summary
assert "uncalibrated" in final_summary.lower()
assert "non-diagnostic" in final_summary.lower()

assert len(final_scores) == 3

print("=" * 80)
print("PHASE 5D — FINAL REGRESSION VERIFICATION")
print("=" * 80)

print("Quantitative measurements: 8/8 PASSED")
print("ED/ES overlays: PASSED")
print("Frozen EF-band classification: PASSED")
print("Confidence and disclaimer wording: PASSED")

print("\n✓ VALIDATED APPLICATION REGRESSION TEST PASSED.")

,measurement,frozen_value,validated_app_value,absolute_difference,passed
0,ED area (mm²),2714.265708,2714.266,0.000292,True
1,ES area (mm²),1520.238767,1520.239,0.000233,True
2,ED length (mm),78.143962,78.144,0.000038,True
3,ES length (mm),62.295982,62.296,0.000018,True
4,EDV (mL),80.025456,80.025,0.000456,True
5,ESV (mL),31.490708,31.491,0.000292,True
6,SV (mL),48.534747,48.535,0.000253,True
7,EF (%),60.649136,60.649,0.000136,True


PHASE 5D — FINAL REGRESSION VERIFICATION
Quantitative measurements: 8/8 PASSED
ED/ES overlays: PASSED
Frozen EF-band classification: PASSED
Confidence and disclaimer wording: PASSED

✓ VALIDATED APPLICATION REGRESSION TEST PASSED.


In [ ]:
# ================================================================
# PHASE 5D — CELL 21 (CORRECTED)
# APPLICATION-LEVEL INVALID SPACING VERIFICATION
# ================================================================

from unittest.mock import patch
import numpy as np

# Load the real ED image once.
real_ed_image, real_ed_spacing = inference.load_nifti_2d(
    str(demo_ed_path)
)

original_loader = inference.load_nifti_2d

def loader_with_invalid_ed_spacing(path):

    if str(path) == str(demo_ed_path):
        return real_ed_image, (0.0, 0.3)

    return original_loader(path)


# Temporarily simulate a loader returning invalid spacing.
# The actual frozen inference module is not modified.
with patch.object(
    inference,
    "load_nifti_2d",
    side_effect=loader_with_invalid_ed_spacing
):

    try:
        run_assistive_analysis(
            str(demo_ed_path),
            str(demo_es_path)
        )

        invalid_spacing_rejected = False
        rejection_message = "Unexpectedly accepted"

    except Exception as exc:

        rejection_message = str(exc)

        invalid_spacing_rejected = (
            "spacing" in rejection_message.lower()
        )


print("=" * 75)
print("APPLICATION-LEVEL SPACING VERIFICATION")
print("=" * 75)

print("Invalid spacing rejected:", invalid_spacing_rejected)
print("Rejection message:", rejection_message)

assert invalid_spacing_rejected, (
    "Application did not reject simulated invalid spacing."
)

print("\n✓ APPLICATION VALIDATION LAYER TEST PASSED.")

APPLICATION-LEVEL SPACING VERIFICATION
Invalid spacing rejected: True
Rejection message: 'Input validation failed: ED: Physical spacing must be positive.'

✓ APPLICATION VALIDATION LAYER TEST PASSED.


In [ ]:
# ================================================================
# PHASE 5D — CELL 22
# FINAL VERIFICATION, DOCUMENTATION AND PHASE 5 COMPLETION
# ================================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 80)
print("PHASE 5D — FINAL VERIFICATION AND COMPLETION")
print("=" * 80)

# ------------------------------------------------
# 1. VERIFY ALL REQUIRED TEST RESULTS
# ------------------------------------------------

assert final_regression_df["passed"].all(), (
    "Final numerical regression verification failed."
)

assert validation_df["passed"].all(), (
    "Input validation tests failed."
)

assert error_test_df["rejected"].all(), (
    "Missing/corrupt input tests failed."
)

assert invalid_spacing_rejected is True, (
    "Corrected application-level spacing test failed."
)

assert numerical_passed, (
    "Original numerical consistency test failed."
)

phase5b_record = (
    PHASE5_ROOT
    / "phase5B_integrated_pipeline"
    / "verification"
    / "PHASE5B_COMPLETE.json"
)

phase5c_record = (
    VERIFICATION_DIR
    / "PHASE5C_COMPLETE.json"
)

assert phase5b_record.exists()
assert phase5c_record.exists()

print("\nAll prerequisite verification checks: PASSED")


# ------------------------------------------------
# 2. SAVE VERIFICATION TABLES
# ------------------------------------------------

PHASE5D_TABLES.mkdir(parents=True, exist_ok=True)
PHASE5D_RECORDS.mkdir(parents=True, exist_ok=True)
PHASE5D_REPORTS.mkdir(parents=True, exist_ok=True)

final_regression_df.to_csv(
    PHASE5D_TABLES / "final_regression_results.csv",
    index=False
)

validation_df.to_csv(
    PHASE5D_TABLES / "input_validation_results.csv",
    index=False
)

error_test_df.to_csv(
    PHASE5D_TABLES / "error_handling_results.csv",
    index=False
)

phase5d_numerical_df.to_csv(
    PHASE5D_TABLES / "initial_numerical_consistency.csv",
    index=False
)

print("Verification tables saved.")


# ------------------------------------------------
# 3. CREATE PHASE 5D COMPLETION RECORD
# ------------------------------------------------

timestamp = datetime.now(timezone.utc).isoformat()

phase5d_record = {
    "phase": "5D",
    "title": "Final System Verification",
    "status": "completed",
    "verified_at_utc": timestamp,

    "verification_results": {
        "initial_numerical_consistency": "passed",
        "validated_application_regression": "passed",
        "segmentation_overlay_generation": "passed",
        "ef_band_output": "passed",
        "non_diagnostic_disclaimer": "passed",
        "uncalibrated_score_wording": "passed",
        "missing_input_handling": "passed",
        "corrupt_input_handling": "passed",
        "direct_input_validation": "passed",
        "mocked_invalid_spacing_rejection": "passed"
    },

    "test_counts": {
        "numerical_measurements_verified": int(
            len(final_regression_df)
        ),
        "direct_validation_cases": int(
            len(validation_df)
        ),
        "missing_or_corrupt_input_cases": int(
            len(error_test_df)
        )
    },

    "spacing_validation": {
        "positive_finite_spacing_required": True,
        "validator_unit_tests_passed": True,
        "application_handler_mock_test_passed": True,
        "raw_nifti_zero_spacing_test_passed": False,
        "raw_nifti_test_explanation": (
            "Nibabel normalized zero-valued pixdim to 1 "
            "before application validation. The original "
            "test therefore did not demonstrate detection "
            "of malformed raw spacing metadata."
        ),
        "known_limitation": (
            "The application validates spacing values "
            "returned by the NIfTI loader. It cannot "
            "reliably detect missing or invalid spacing "
            "that has already been repaired or replaced "
            "with defaults by the reader."
        )
    },

    "scientific_scope": {
        "application_type": "Research assistive prototype",
        "clinical_validation": False,
        "diagnostic_use": False,
        "frozen_model_retraining": False,
        "evaluation_scope": (
            "Previously established CAMUS patient-level "
            "evaluation partition; not independent "
            "external clinical validation."
        )
    }
}

phase5d_record_path = (
    PHASE5D_RECORDS / "PHASE5D_COMPLETE.json"
)

with open(phase5d_record_path, "w") as f:
    json.dump(phase5d_record, f, indent=2)

print("Phase 5D completion record saved.")


# ------------------------------------------------
# 4. CREATE FINAL HUMAN-READABLE REPORT
# ------------------------------------------------

report = f"""
PHASE 5 — FINAL SYSTEM VERIFICATION REPORT
=========================================

Project:
Computer Vision Based Assistive Application
for Analyzing ECHO Scans — LV (A4C)

Verification date (UTC):
{timestamp}

PHASE STATUS
------------
Phase 5A: System specification completed
Phase 5B: Frozen integrated inference completed
Phase 5C: Gradio assistive application completed
Phase 5D: Final verification completed

IMPLEMENTED WORKFLOW
--------------------
1. Upload paired ED and ES A4C NIfTI images.
2. Validate image content and returned physical spacing.
3. Perform frozen LV segmentation.
4. Generate ED and ES LV contour overlays.
5. Calculate LV area, length, EDV, ESV, SV and EF.
6. Perform frozen EF-band classification.
7. Display uncalibrated class scores.
8. Present non-diagnostic research disclaimer.

FINAL VERIFICATION RESULTS
--------------------------
Initial numerical consistency: PASSED
Validated application regression: PASSED
ED/ES overlays: PASSED
EF-band classification: PASSED
Missing input handling: PASSED
Corrupt NIfTI handling: PASSED
Direct validation tests: PASSED
Mocked invalid-spacing handler test: PASSED

Numerical measurements verified:
{len(final_regression_df)}

Direct validation cases:
{len(validation_df)}

Missing/corrupt input cases:
{len(error_test_df)}

IMPORTANT LIMITATIONS
---------------------
1. The system is a non-diagnostic research prototype.

2. Classification scores are uncalibrated and must
   not be interpreted as clinical probabilities.

3. The application requires correctly identified
   paired A4C ED/ES images.

4. Pixel spacing validation operates on values
   returned by the NIfTI loader. Nibabel may repair
   zero-valued spacing metadata before validation,
   so detection of malformed original metadata
   is not guaranteed.

5. The evaluation uses the previously established
   CAMUS partition and is not external clinical
   validation.

6. No frozen segmentation or classification model
   was retrained during Phase 5.

FINAL STATUS
------------
Phase 5 implementation and documented verification
completed within the defined research prototype scope.
"""

report_path = (
    PHASE5D_REPORTS / "PHASE5_FINAL_VERIFICATION_REPORT.txt"
)

report_path.write_text(report, encoding="utf-8")

print("Final verification report saved.")


# ------------------------------------------------
# 5. CREATE OVERALL PHASE 5 COMPLETION RECORD
# ------------------------------------------------

overall_record = {
    "project": (
        "Computer Vision Based Assistive Application "
        "for Analyzing ECHO Scans — LV A4C"
    ),

    "phase": "5",
    "status": "completed",
    "completed_at_utc": timestamp,

    "phases": {
        "5A_system_specification": "completed",
        "5B_integrated_frozen_inference": "completed",
        "5C_assistive_application": "completed",
        "5D_final_verification": "completed"
    },

    "system": {
        "interface": "Gradio",
        "segmentation": "Frozen Large U-Net",
        "quantification": "Frozen Phase 2 geometry",
        "classification": (
            "Frozen StandardScaler + Logistic Regression"
        ),
        "output": (
            "LV segmentation overlays, quantitative "
            "measurements, EF-band classification, "
            "uncalibrated scores and disclaimer"
        )
    },

    "verification": {
        "phase5B_reproduction": "passed",
        "phase5C_demonstration": "passed",
        "phase5D_application_tests": "passed",
        "known_spacing_limitation_documented": True
    },

    "clinical_validation": False,
    "diagnostic_use": False,
    "models_retrained_in_phase5": False,

    "final_verification_record": str(
        phase5d_record_path
    ),

    "final_report": str(report_path)
}

overall_record_path = (
    PHASE5_ROOT / "PHASE5_COMPLETE.json"
)

with open(overall_record_path, "w") as f:
    json.dump(overall_record, f, indent=2)


# ------------------------------------------------
# 6. VERIFY FILES EXIST
# ------------------------------------------------

required_files = [
    phase5d_record_path,
    overall_record_path,
    report_path,
    PHASE5D_TABLES / "final_regression_results.csv",
    PHASE5D_TABLES / "input_validation_results.csv",
    PHASE5D_TABLES / "error_handling_results.csv",
    PHASE5D_TABLES / "initial_numerical_consistency.csv"
]

for file_path in required_files:
    assert file_path.exists(), (
        f"Missing output file: {file_path}"
    )


# ------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 80)
print("PHASE 5 — FINAL COMPLETION SUMMARY")
print("=" * 80)

print("\nPhase 5A: COMPLETED")
print("Phase 5B: COMPLETED")
print("Phase 5C: COMPLETED")
print("Phase 5D: COMPLETED")

print("\nFinal verification files:")
print("1.", phase5d_record_path)
print("2.", overall_record_path)
print("3.", report_path)

print("\nAll required verification files: FOUND")

print("\n" + "=" * 80)
print("✓ PHASE 5 COMPLETED AND DOCUMENTED")
print("=" * 80)

PHASE 5D — FINAL VERIFICATION AND COMPLETION

All prerequisite verification checks: PASSED
Verification tables saved.
Phase 5D completion record saved.
Final verification report saved.

PHASE 5 — FINAL COMPLETION SUMMARY

Phase 5A: COMPLETED
Phase 5B: COMPLETED
Phase 5C: COMPLETED
Phase 5D: COMPLETED

Final verification files:
1. /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5D_final_verification/verification/PHASE5D_COMPLETE.json
2. /content/drive/MyDrive/CAMUS_FINAL/phase5_application/PHASE5_COMPLETE.json
3. /content/drive/MyDrive/CAMUS_FINAL/phase5_application/phase5D_final_verification/reports/PHASE5_FINAL_VERIFICATION_REPORT.txt

All required verification files: FOUND

✓ PHASE 5 COMPLETED AND DOCUMENTED


In [ ]:
# Save a frozen snapshot of the completed notebook's source location
from pathlib import Path
import json
from datetime import datetime, timezone

snapshot_path = (
    PROJECT_ROOT
    / "phase5_application"
    / "phase5C_assistive_application"
    / "verification"
    / "PHASE5_FINAL_SNAPSHOT.json"
)

snapshot = {
    "status": "frozen_for_phase5E",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "Phase5C_Assistive_Application_FINAL.ipynb",
    "phase5_completed": True,
    "validated_handler": True,
    "frozen_segmentation_model_unchanged": True,
    "frozen_classifier_unchanged": True,
    "next_stage": "Phase5E_quality_and_interpretation"
}

snapshot_path.write_text(
    json.dumps(snapshot, indent=2),
    encoding="utf-8"
)

print("✓ Phase 5 snapshot record saved.")

✓ Phase 5 snapshot record saved.
